# 🚀 Water Meter AI - Google Colab Training

**Author:** Arsenius Purbandono  
**Objective:** Train YOLOv8-OBB model on Google Colab (Free GPU)

---

## 🎯 Colab Setup Checklist
- [ ] Runtime > Change runtime type > **T4 GPU**
- [ ] Upload your dataset or mount Google Drive
- [ ] Run all cells in order

---

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q ultralytics roboflow loguru pyyaml

In [ ]:
# Verify installation
import ultralytics
ultralytics.checks()

import torch
print(f"\n🔥 PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive (Optional)

In [ ]:
# Uncomment if you want to save results to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Set output directory
# output_dir = '/content/drive/MyDrive/water-meter-ai'
# !mkdir -p "{output_dir}"

## 3. Download Dataset

**Option A:** Upload your dataset ZIP file  
**Option B:** Use Roboflow API  
**Option C:** Mount Google Drive with dataset

In [ ]:
# Option A: Upload dataset ZIP
# from google.colab import files
# uploaded = files.upload()
# !unzip -q your-dataset.zip -d /content/dataset

# Option B: Roboflow (if using Roboflow)
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# dataset = project.version(1).download("yolov8-obb")

# Option C: Google Drive
# !cp -r /content/drive/MyDrive/water-meter-dataset /content/dataset

# Set dataset path
dataset_path = '/content/dataset'  # Adjust this path
print(f"Dataset path: {dataset_path}")

## 4. Dataset Verification

In [ ]:
# Check dataset structure
import os
from pathlib import Path

data_yaml = Path(dataset_path) / 'data.yaml'
print(f"Data YAML exists: {data_yaml.exists()}")

if data_yaml.exists():
    import yaml
    with open(data_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print("\n📊 Dataset Configuration:")
    print(f"  Train: {data_config.get('train')}")
    print(f"  Val: {data_config.get('val')}")
    print(f"  Test: {data_config.get('test')}")
    print(f"  Classes: {data_config.get('nc')}")
    print(f"  Names: {data_config.get('names')}")
    
    # Count images
    train_images = len(list(Path(dataset_path).glob('train/images/*.jpg')))
    val_images = len(list(Path(dataset_path).glob('valid/images/*.jpg')))
    test_images = len(list(Path(dataset_path).glob('test/images/*.jpg')))
    
    print(f"\n📁 Image Counts:")
    print(f"  Train: {train_images:,} images")
    print(f"  Valid: {val_images:,} images")
    print(f"  Test: {test_images:,} images")
    print(f"  Total: {train_images + val_images + test_images:,} images")

## 5. Training Configuration

In [ ]:
# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 16  # Adjust based on GPU memory
IMAGE_SIZE = 512
MODEL_TYPE = 'yolov8n-obb'  # Options: n, s, m, l, x
PATIENCE = 20  # Early stopping patience

print("🎯 Training Configuration:")
print(f"  Model: {MODEL_TYPE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMAGE_SIZE}")
print(f"  Patience: {PATIENCE}")

## 6. Initialize Model

In [ ]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO(f'{MODEL_TYPE}.pt')
print(f"✅ Model {MODEL_TYPE} loaded!")

## 7. Start Training 🚀

In [ ]:
# Start training
results = model.train(
    # Data
    data=str(data_yaml),
    imgsz=IMAGE_SIZE,
    
    # Training
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    
    # Augmentation (aligned with blueprint)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,      # ±15° rotation
    translate=0.1,
    scale=0.5,
    shear=10.0,        # ±10° shear
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    
    # Hardware
    device=0,          # Use first GPU
    workers=8,
    amp=True,          # Automatic Mixed Precision
    
    # Logging
    project='water-meter-ai',
    name='colab-exp',
    exist_ok=True,
    plots=True,
    verbose=True,
    
    # Misc
    seed=42,
)

## 8. Validation & Metrics

In [ ]:
# Validate the trained model
metrics = model.val()

print("\n" + "="*60)
print("📊 FINAL VALIDATION METRICS")
print("="*60)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print("="*60)

# Check if meets targets
target_map50 = 0.90  # 90% target
if metrics.box.map50 >= target_map50:
    print(f"\n✅ TARGET MET! mAP50 >= {target_map50:.0%}")
else:
    print(f"\n⚠️ Target not met. Current: {metrics.box.map50:.2%}, Target: {target_map50:.0%}")

## 9. Visualize Results

In [ ]:
# Display training plots
from IPython.display import Image, display

# Results plot
results_plot = 'water-meter-ai/colab-exp/results.png'
if Path(results_plot).exists():
    print("📈 Training Results:")
    display(Image(filename=results_plot))

# Confusion matrix
confusion_matrix = 'water-meter-ai/colab-exp/confusion_matrix.png'
if Path(confusion_matrix).exists():
    print("\n📊 Confusion Matrix:")
    display(Image(filename=confusion_matrix))

## 10. Export to TFLite (Mobile Deployment)

In [ ]:
# Export to TFLite with INT8 quantization
print("📦 Exporting to TFLite (INT8 Quantized)...")

tflite_path = model.export(
    format='tflite',
    imgsz=IMAGE_SIZE,
    int8=True,          # INT8 quantization
    optimize=True,      # Optimize for mobile
    simplify=True,      # Simplify model
)

print(f"\n✅ TFLite model exported: {tflite_path}")

# Check model size
if Path(tflite_path).exists():
    size_mb = Path(tflite_path).stat().st_size / (1024 * 1024)
    print(f"📦 Model Size: {size_mb:.2f} MB")
    
    if size_mb < 10:
        print("✅ Model size meets target (<10 MB)")
    else:
        print(f"⚠️ Model size exceeds target (10 MB)")

## 11. Test Inference

In [ ]:
# Test on sample images
test_images = list(Path(dataset_path).glob('test/images/*.jpg'))[:5]

if test_images:
    print(f"🧪 Testing on {len(test_images)} images...\n")
    
    for img_path in test_images:
        results = model(str(img_path))
        
        # Display image with detections
        print(f"Image: {img_path.name}")
        results[0].show()
        
        # Print detections
        boxes = results[0].obb
        if boxes is not None:
            print(f"  Detections: {len(boxes)} objects")
            for i, box in enumerate(boxes):
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                print(f"    {i+1}. Class: {cls}, Confidence: {conf:.2f}")
        print("\n" + "-"*60 + "\n")
else:
    print("❌ No test images found!")

## 12. Download Results

In [ ]:
# Zip results for download
!zip -r water-meter-results.zip water-meter-ai/colab-exp/

# Download
from google.colab import files
files.download('water-meter-results.zip')
print("\n✅ Results downloaded!")

## 13. Save to Google Drive (Optional)

In [ ]:
# Copy results to Google Drive
# !cp -r water-meter-ai/colab-exp /content/drive/MyDrive/water-meter-ai/
# !cp {tflite_path} /content/drive/MyDrive/water-meter-ai/
# print("✅ Results saved to Google Drive!")

---

## ✅ Training Complete!

### 📋 Next Steps:

1. **Download Results:**
   - Best weights: `best.pt`
   - TFLite model: `best.tflite`
   - Training plots & metrics

2. **Model Integration:**
   - Integrate `.tflite` into Flutter app
   - Test on real mobile devices
   - Benchmark inference speed

3. **Model Improvement (if needed):**
   - Analyze confusion matrix
   - Add more training data
   - Try larger model (yolov8s-obb, yolov8m-obb)
   - Adjust augmentation parameters

4. **Deployment:**
   - Create Flutter app with TFLite interpreter
   - Test in production environment
   - Monitor performance metrics

---

**Author:** Arsenius Purbandono  
**Project:** Water Meter AI - Digital Transformation for Legacy Infrastructure